In [2]:
import pandas as pd

In [4]:
import pandas as pd

print(pd.__version__)

3.0.5


In [5]:
fake_df = pd.read_csv("../data/raw/Fake.csv")
true_df = pd.read_csv("../data/raw/True.csv")

In [6]:
print("FAKE NEWS DATASET:")
print(fake_df.shape)

print("\nTRUE NEWS DATASET:")
print(true_df.shape)

FAKE NEWS DATASET:
(23481, 4)

TRUE NEWS DATASET:
(21417, 4)


In [7]:
print("FAKE DATA COLUMNS:")
print(fake_df.columns)

print("\nTRUE DATA COLUMNS:")
print(true_df.columns)

FAKE DATA COLUMNS:
Index(['title', 'text', 'subject', 'date'], dtype='str')

TRUE DATA COLUMNS:
Index(['title', 'text', 'subject', 'date'], dtype='str')


In [8]:
print("FAKE NEWS SAMPLE:")
display(fake_df.head(3))

print("\nTRUE NEWS SAMPLE:")
display(true_df.head(3))

FAKE NEWS SAMPLE:


,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"



TRUE NEWS SAMPLE:


,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"


In [9]:
fake_df["label"] = 0
true_df["label"] = 1

print("Fake labels:", fake_df["label"].unique())
print("True labels:", true_df["label"].unique())

Fake labels: [0]
True labels: [1]


In [10]:
# Combine fake and true news datasets

df = pd.concat([fake_df, true_df], ignore_index=True)

print("COMBINED DATASET SHAPE:")
print(df.shape)

print("\nLABEL DISTRIBUTION:")
print(df["label"].value_counts())

print("\nDATASET PREVIEW:")
display(df.head())

COMBINED DATASET SHAPE:
(44898, 5)

LABEL DISTRIBUTION:
label
0    23481
1    21417
Name: count, dtype: int64

DATASET PREVIEW:


,title,text,subject,date,label
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0


In [11]:
print("MISSING VALUES:")
print(df.isnull().sum())


MISSING VALUES:
title      0
text       0
subject    0
date       0
label      0
dtype: int64


In [12]:
print("DUPLICATE ROWS:", df.duplicated().sum())

DUPLICATE ROWS: 209


In [13]:
# Remove duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("NEW DATASET SHAPE:", df.shape)
print("DUPLICATE ROWS REMAINING:", df.duplicated().sum())

NEW DATASET SHAPE: (44689, 5)
DUPLICATE ROWS REMAINING: 0


In [14]:
# Combine title and article text
df["content"] = df["title"].astype(str) + " " + df["text"].astype(str)

# Check the result
print("DATASET SHAPE:", df.shape)

print("\nCONTENT SAMPLE:")
print(df["content"].iloc[0][:500])

DATASET SHAPE: (44689, 6)

CONTENT SAMPLE:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger and smarter, I want to wish all of my friends, supporters, enemies, haters, and even the very dishonest Fake News Media, 


In [15]:
import re

def clean_text(text):
    # Convert to lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r"http\S+|www\S+", "", text)
    
    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()
    
    return text

# Apply cleaning
df["clean_content"] = df["content"].apply(clean_text)

print("ORIGINAL:")
print(df["content"].iloc[0][:300])

print("\nCLEANED:")
print(df["clean_content"].iloc[0][:300])

ORIGINAL:
 Donald Trump Sends Out Embarrassing New Year’s Eve Message; This is Disturbing Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had j

CLEANED:
donald trump sends out embarrassing new year’s eve message; this is disturbing donald trump just couldn t wish all americans a happy new year and leave it at that. instead, he had to give a shout out to his enemies, haters and the very dishonest fake news media. the former reality show star had just


In [16]:
# Features and target
X = df["clean_content"]
y = df["label"]

print("FEATURES SHAPE:", X.shape)
print("LABELS SHAPE:", y.shape)

print("\nLABEL DISTRIBUTION:")
print(y.value_counts())

FEATURES SHAPE: (44689,)
LABELS SHAPE: (44689,)

LABEL DISTRIBUTION:
label
0    23478
1    21211
Name: count, dtype: int64


In [17]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("TRAINING FEATURES:", X_train.shape)
print("TESTING FEATURES:", X_test.shape)

print("\nTRAINING LABELS:", y_train.shape)
print("TESTING LABELS:", y_test.shape)

TRAINING FEATURES: (35751,)
TESTING FEATURES: (8938,)

TRAINING LABELS: (35751,)
TESTING LABELS: (8938,)


In [18]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    stop_words="english",
    max_df=0.7,
    max_features=50000
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("TRAINING TF-IDF SHAPE:", X_train_tfidf.shape)
print("TESTING TF-IDF SHAPE:", X_test_tfidf.shape)

TRAINING TF-IDF SHAPE: (35751, 50000)
TESTING TF-IDF SHAPE: (8938, 50000)


In [19]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

model.fit(X_train_tfidf, y_train)

print("Model training completed successfully!")

Model training completed successfully!


In [20]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions on unseen test data
y_pred = model.predict(X_test_tfidf)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)

print("MODEL ACCURACY:", accuracy)
print(f"MODEL ACCURACY: {accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Fake", "Real"]
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, y_pred))

MODEL ACCURACY: 0.9862385321100917
MODEL ACCURACY: 98.62%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      4696
        Real       0.98      0.99      0.99      4242

    accuracy                           0.99      8938
   macro avg       0.99      0.99      0.99      8938
weighted avg       0.99      0.99      0.99      8938


CONFUSION MATRIX:

[[4623   73]
 [  50 4192]]


In [21]:
def predict_news(news_text):
    
    # Clean the input text
    cleaned_text = clean_text(news_text)
    
    # Convert text using the trained TF-IDF vectorizer
    text_tfidf = tfidf.transform([cleaned_text])
    
    # Make prediction
    prediction = model.predict(text_tfidf)[0]
    
    # Get confidence probabilities
    probabilities = model.predict_proba(text_tfidf)[0]
    
    label = "REAL NEWS" if prediction == 1 else "FAKE NEWS"
    
    confidence = probabilities[prediction] * 100
    
    print("Prediction:", label)
    print(f"Confidence: {confidence:.2f}%")
    
    return prediction

In [22]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create the model
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train_tfidf, y_train)

# Make predictions
nb_pred = nb_model.predict(X_test_tfidf)

# Calculate accuracy
nb_accuracy = accuracy_score(y_test, nb_pred)

print("NAIVE BAYES ACCURACY:", nb_accuracy)
print(f"NAIVE BAYES ACCURACY: {nb_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(
    y_test,
    nb_pred,
    target_names=["Fake", "Real"]
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, nb_pred))

NAIVE BAYES ACCURACY: 0.935444170955471
NAIVE BAYES ACCURACY: 93.54%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.94      0.94      0.94      4696
        Real       0.93      0.93      0.93      4242

    accuracy                           0.94      8938
   macro avg       0.94      0.94      0.94      8938
weighted avg       0.94      0.94      0.94      8938


CONFUSION MATRIX:

[[4411  285]
 [ 292 3950]]


In [23]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Create Linear SVM model
svm_model = LinearSVC(random_state=42)

# Train the model
svm_model.fit(X_train_tfidf, y_train)

# Make predictions
svm_pred = svm_model.predict(X_test_tfidf)

# Calculate accuracy
svm_accuracy = accuracy_score(y_test, svm_pred)

print("LINEAR SVM ACCURACY:", svm_accuracy)
print(f"LINEAR SVM ACCURACY: {svm_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(
    y_test,
    svm_pred,
    target_names=["Fake", "Real"]
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, svm_pred))

LINEAR SVM ACCURACY: 0.9966435444170956
LINEAR SVM ACCURACY: 99.66%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      4696
        Real       1.00      1.00      1.00      4242

    accuracy                           1.00      8938
   macro avg       1.00      1.00      1.00      8938
weighted avg       1.00      1.00      1.00      8938


CONFUSION MATRIX:

[[4683   13]
 [  17 4225]]


In [24]:
# Check whether any cleaned articles appear in both training and testing sets

train_articles = set(X_train)
test_articles = set(X_test)

overlap = train_articles.intersection(test_articles)

print("TRAINING ARTICLES:", len(train_articles))
print("TESTING ARTICLES:", len(test_articles))
print("OVERLAPPING ARTICLES:", len(overlap))

if len(overlap) == 0:
    print("\n✅ No exact overlap found between training and testing data.")
else:
    print("\n⚠️ WARNING: Overlapping articles found!")
    

TRAINING ARTICLES: 32136
TESTING ARTICLES: 8695
OVERLAPPING ARTICLES: 1735

⚠️ WARNING: Overlapping articles found!


In [25]:
print("DATASET SHAPE BEFORE CLEAN_CONTENT DEDUPLICATION:", df.shape)

# Remove duplicate cleaned articles
df = df.drop_duplicates(subset=["clean_content"]).reset_index(drop=True)

print("DATASET SHAPE AFTER CLEAN_CONTENT DEDUPLICATION:", df.shape)

print("\nDUPLICATE CLEAN_CONTENT REMAINING:")
print(df["clean_content"].duplicated().sum())

print("\nLABEL DISTRIBUTION:")
print(df["label"].value_counts())

DATASET SHAPE BEFORE CLEAN_CONTENT DEDUPLICATION: (44689, 7)
DATASET SHAPE AFTER CLEAN_CONTENT DEDUPLICATION: (39096, 7)

DUPLICATE CLEAN_CONTENT REMAINING:
0

LABEL DISTRIBUTION:
label
1    21195
0    17901
Name: count, dtype: int64


In [26]:
# Features and target
X = df["clean_content"]
y = df["label"]

print("FEATURES SHAPE:", X.shape)
print("LABELS SHAPE:", y.shape)

print("\nLABEL DISTRIBUTION:")
print(y.value_counts())


FEATURES SHAPE: (39096,)
LABELS SHAPE: (39096,)

LABEL DISTRIBUTION:
label
1    21195
0    17901
Name: count, dtype: int64


In [27]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("TRAINING FEATURES:", X_train.shape)
print("TESTING FEATURES:", X_test.shape)

print("\nTRAINING LABELS:", y_train.shape)
print("TESTING LABELS:", y_test.shape)

print("\nTRAINING LABEL DISTRIBUTION:")
print(y_train.value_counts())

print("\nTESTING LABEL DISTRIBUTION:")
print(y_test.value_counts())

TRAINING FEATURES: (31276,)
TESTING FEATURES: (7820,)

TRAINING LABELS: (31276,)
TESTING LABELS: (7820,)

TRAINING LABEL DISTRIBUTION:
label
1    16956
0    14320
Name: count, dtype: int64

TESTING LABEL DISTRIBUTION:
label
1    4239
0    3581
Name: count, dtype: int64


In [28]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    stop_words="english",
    ngram_range=(1, 2)
)

# Learn vocabulary ONLY from training data
X_train_tfidf = tfidf.fit_transform(X_train)

# Transform testing data using the same vocabulary
X_test_tfidf = tfidf.transform(X_test)

print("TRAINING TF-IDF SHAPE:", X_train_tfidf.shape)
print("TESTING TF-IDF SHAPE:", X_test_tfidf.shape)

TRAINING TF-IDF SHAPE: (31276, 50000)
TESTING TF-IDF SHAPE: (7820, 50000)


In [29]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

logistic_model.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed successfully!")

Logistic Regression training completed successfully!


In [30]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=2000,
    random_state=42
)

logistic_model.fit(X_train_tfidf, y_train)

print("Logistic Regression training completed successfully!")

Logistic Regression training completed successfully!


In [31]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions
logistic_pred = logistic_model.predict(X_test_tfidf)

# Calculate accuracy
logistic_accuracy = accuracy_score(y_test, logistic_pred)

print("LOGISTIC REGRESSION ACCURACY:", logistic_accuracy)
print(f"LOGISTIC REGRESSION ACCURACY: {logistic_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:\n")

print(classification_report(
    y_test,
    logistic_pred,
    target_names=["Fake", "Real"]
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, logistic_pred))

LOGISTIC REGRESSION ACCURACY: 0.9867007672634271
LOGISTIC REGRESSION ACCURACY: 98.67%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.99      0.98      0.99      3581
        Real       0.98      0.99      0.99      4239

    accuracy                           0.99      7820
   macro avg       0.99      0.99      0.99      7820
weighted avg       0.99      0.99      0.99      7820


CONFUSION MATRIX:

[[3511   70]
 [  34 4205]]


In [32]:
from sklearn.naive_bayes import MultinomialNB

# Create the Naive Bayes model
nb_model = MultinomialNB()

# Train the model
nb_model.fit(X_train_tfidf, y_train)

print("Naive Bayes training completed successfully!")

Naive Bayes training completed successfully!


In [33]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions
nb_pred = nb_model.predict(X_test_tfidf)

# Calculate accuracy
nb_accuracy = accuracy_score(y_test, nb_pred)

print("NAIVE BAYES ACCURACY:", nb_accuracy)
print(f"NAIVE BAYES ACCURACY: {nb_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:\n")

print(classification_report(
    y_test,
    nb_pred,
    target_names=["Fake", "Real"]
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, nb_pred))

NAIVE BAYES ACCURACY: 0.9525575447570332
NAIVE BAYES ACCURACY: 95.26%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.95      0.95      0.95      3581
        Real       0.96      0.96      0.96      4239

    accuracy                           0.95      7820
   macro avg       0.95      0.95      0.95      7820
weighted avg       0.95      0.95      0.95      7820


CONFUSION MATRIX:

[[3399  182]
 [ 189 4050]]


In [34]:
from sklearn.svm import LinearSVC

# Create the Linear SVM model
svm_model = LinearSVC(
    random_state=42,
    max_iter=5000
)

# Train the model
svm_model.fit(X_train_tfidf, y_train)

print("Linear SVM training completed successfully!")

Linear SVM training completed successfully!


In [35]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions on the test data
svm_pred = svm_model.predict(X_test_tfidf)

# Calculate accuracy
svm_accuracy = accuracy_score(y_test, svm_pred)

print("LINEAR SVM ACCURACY:", svm_accuracy)
print(f"LINEAR SVM ACCURACY: {svm_accuracy * 100:.2f}%")

# Classification report
print("\nCLASSIFICATION REPORT:\n")
print(classification_report(
    y_test,
    svm_pred,
    target_names=["Fake", "Real"]
))

# Confusion matrix
print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, svm_pred))

LINEAR SVM ACCURACY: 0.9941176470588236
LINEAR SVM ACCURACY: 99.41%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       1.00      0.99      0.99      3581
        Real       0.99      1.00      0.99      4239

    accuracy                           0.99      7820
   macro avg       0.99      0.99      0.99      7820
weighted avg       0.99      0.99      0.99      7820


CONFUSION MATRIX:

[[3548   33]
 [  13 4226]]


In [36]:
# Check exact text overlap between training and testing data

train_articles = set(X_train)
test_articles = set(X_test)

overlap = train_articles.intersection(test_articles)

print("TRAINING ARTICLES:", len(train_articles))
print("TESTING ARTICLES:", len(test_articles))
print("OVERLAPPING ARTICLES:", len(overlap))

if len(overlap) == 0:
    print("\n✅ SUCCESS: No exact overlap between training and testing data!")
else:
    print(f"\n⚠️ WARNING: {len(overlap)} overlapping articles found!")

TRAINING ARTICLES: 31276
TESTING ARTICLES: 7820
OVERLAPPING ARTICLES: 0

✅ SUCCESS: No exact overlap between training and testing data!


In [37]:
from sklearn.model_selection import cross_val_score
from sklearn.svm import LinearSVC

# Create a fresh Linear SVM model
cv_svm = LinearSVC(
    random_state=42,
    max_iter=5000
)

# Perform 5-fold cross-validation
cv_scores = cross_val_score(
    cv_svm,
    X_train_tfidf,
    y_train,
    cv=5,
    scoring="accuracy"
)

print("CROSS-VALIDATION SCORES:")
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score * 100:.2f}%")

print("\nAVERAGE CV ACCURACY:", f"{cv_scores.mean() * 100:.2f}%")
print("STANDARD DEVIATION:", f"{cv_scores.std() * 100:.2f}%")

CROSS-VALIDATION SCORES:
Fold 1: 99.42%
Fold 2: 99.44%
Fold 3: 99.26%
Fold 4: 99.38%
Fold 5: 99.17%

AVERAGE CV ACCURACY: 99.33%
STANDARD DEVIATION: 0.10%


In [38]:
from sklearn.ensemble import RandomForestClassifier

# Create Random Forest model
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Train the model
rf_model.fit(X_train_tfidf, y_train)

print("Random Forest training completed successfully!")

Random Forest training completed successfully!


In [39]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Make predictions
rf_pred = rf_model.predict(X_test_tfidf)

# Calculate accuracy
rf_accuracy = accuracy_score(y_test, rf_pred)

print("RANDOM FOREST ACCURACY:", rf_accuracy)
print(f"RANDOM FOREST ACCURACY: {rf_accuracy * 100:.2f}%")

# Classification report
print("\nCLASSIFICATION REPORT:\n")
print(classification_report(
    y_test,
    rf_pred,
    target_names=["Fake", "Real"]
))

# Confusion matrix
print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(y_test, rf_pred))

RANDOM FOREST ACCURACY: 0.9937340153452685
RANDOM FOREST ACCURACY: 99.37%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       1.00      0.99      0.99      3581
        Real       0.99      1.00      0.99      4239

    accuracy                           0.99      7820
   macro avg       0.99      0.99      0.99      7820
weighted avg       0.99      0.99      0.99      7820


CONFUSION MATRIX:

[[3545   36]
 [  13 4226]]


In [40]:
# Prepare IFND external test dataset

ifnd_test = df[["Statement", "Label"]].copy()

# Rename columns for consistency
ifnd_test.columns = ["text", "label"]

# Convert labels to our model format
ifnd_test["label"] = ifnd_test["label"].map({
    "Fake": 0,
    "TRUE": 1
})

# Check for any invalid labels
print("DATASET SHAPE:", ifnd_test.shape)

print("\nLABEL DISTRIBUTION:")
print(ifnd_test["label"].value_counts())

print("\nMISSING VALUES:")
print(ifnd_test.isnull().sum())

# Remove any rows with missing values just for safety
ifnd_test = ifnd_test.dropna()

print("\nFINAL TEST DATASET SHAPE:", ifnd_test.shape)


KeyError: "None of [Index(['Statement', 'Label'], dtype='str')] are in the [columns]"

In [41]:
import pandas as pd

# Load IFND external dataset
ifnd_df = pd.read_csv(
    "../data/raw/IFND.csv",
    encoding="latin1"
)

print("IFND DATASET SHAPE:", ifnd_df.shape)

print("\nCOLUMNS:")
print(ifnd_df.columns.tolist())

print("\nLABEL DISTRIBUTION:")
print(ifnd_df["Label"].value_counts())

print("\nFIRST 5 ROWS:")
print(ifnd_df[["Statement", "Label"]].head())

IFND DATASET SHAPE: (56714, 7)

COLUMNS:
['id', 'Statement', 'Image', 'Web', 'Category', 'Date', 'Label']

LABEL DISTRIBUTION:
Label
TRUE    37800
Fake    18914
Name: count, dtype: int64

FIRST 5 ROWS:
                                           Statement Label
0  WHO praises India's Aarogya Setu app, says it ...  TRUE
1  In Delhi, Deputy US Secretary of State Stephen...  TRUE
2  LAC tensions: China's strategy behind delibera...  TRUE
3  India has signed 250 documents on Space cooper...  TRUE
4  Tamil Nadu chief minister's mother passes away...  TRUE


In [42]:
# Create a clean external test dataset

ifnd_test = ifnd_df[["Statement", "Label"]].copy()

# Rename columns to match our pipeline
ifnd_test.columns = ["text", "label"]

# Convert labels to our project's format
# Fake = 0
# TRUE = 1 (Real)

ifnd_test["label"] = ifnd_test["label"].map({
    "Fake": 0,
    "TRUE": 1
})

# Remove missing or invalid rows
ifnd_test = ifnd_test.dropna()

# Ensure text is string
ifnd_test["text"] = ifnd_test["text"].astype(str)

print("FINAL IFND TEST DATASET SHAPE:", ifnd_test.shape)

print("\nLABEL DISTRIBUTION:")
print(ifnd_test["label"].value_counts())

print("\nVERIFY LABELS:")
print("Unique labels:", sorted(ifnd_test["label"].unique()))

FINAL IFND TEST DATASET SHAPE: (56714, 2)

LABEL DISTRIBUTION:
label
1    37800
0    18914
Name: count, dtype: int64

VERIFY LABELS:
Unique labels: [np.int64(0), np.int64(1)]


In [43]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Transforming IFND text using existing TF-IDF vectorizer...")

# Transform external data using the SAME TF-IDF vocabulary
X_ifnd_tfidf = tfidf.transform(ifnd_test["text"])

print("TF-IDF transformation completed!")
print("IFND TF-IDF shape:", X_ifnd_tfidf.shape)


print("\nMaking predictions using the trained Linear SVM...")

# Predict using the already-trained SVM model
ifnd_predictions = svm_model.predict(X_ifnd_tfidf)


# Calculate accuracy
ifnd_accuracy = accuracy_score(
    ifnd_test["label"],
    ifnd_predictions
)

print("\n" + "=" * 55)
print("EXTERNAL VALIDATION — LINEAR SVM ON IFND DATASET")
print("=" * 55)

print(f"\nACCURACY: {ifnd_accuracy:.6f}")
print(f"ACCURACY PERCENTAGE: {ifnd_accuracy * 100:.2f}%")


print("\nCLASSIFICATION REPORT:\n")

print(
    classification_report(
        ifnd_test["label"],
        ifnd_predictions,
        target_names=["Fake", "Real"]
    )
)


print("\nCONFUSION MATRIX:\n")

print(
    confusion_matrix(
        ifnd_test["label"],
        ifnd_predictions
    )
)

Transforming IFND text using existing TF-IDF vectorizer...
TF-IDF transformation completed!
IFND TF-IDF shape: (56714, 50000)

Making predictions using the trained Linear SVM...

EXTERNAL VALIDATION — LINEAR SVM ON IFND DATASET

ACCURACY: 0.407095
ACCURACY PERCENTAGE: 40.71%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.36      0.97      0.52     18914
        Real       0.90      0.12      0.22     37800

    accuracy                           0.41     56714
   macro avg       0.63      0.55      0.37     56714
weighted avg       0.72      0.41      0.32     56714


CONFUSION MATRIX:

[[18368   546]
 [33080  4720]]


In [44]:
print("Original IFND Labels:")
print(ifnd_df["Label"].value_counts())

# Create numeric labels explicitly
ifnd_df["numeric_label"] = ifnd_df["Label"].map({
    "Fake": 0,
    "TRUE": 1
})

print("\nNumeric Label Distribution:")
print(ifnd_df["numeric_label"].value_counts())

print("\nMissing labels:")
print(ifnd_df["numeric_label"].isna().sum())

Original IFND Labels:
Label
TRUE    37800
Fake    18914
Name: count, dtype: int64

Numeric Label Distribution:
numeric_label
1    37800
0    18914
Name: count, dtype: int64

Missing labels:
0


In [45]:
# Create a DataFrame to inspect actual vs predicted results

inspection_df = pd.DataFrame({
    "Statement": ifnd_df["Statement"],
    "Actual_Label": ifnd_df["numeric_label"],
    "Predicted_Label": ifnd_predictions
})

# Convert numeric labels into readable labels
inspection_df["Actual"] = inspection_df["Actual_Label"].map({
    0: "Fake",
    1: "Real"
})

inspection_df["Predicted"] = inspection_df["Predicted_Label"].map({
    0: "Fake",
    1: "Real"
})

# Show random examples
print(
    inspection_df[
        ["Statement", "Actual", "Predicted"]
    ].sample(10, random_state=42).to_string(index=False)
)

                                                                                           Statement Actual Predicted
                          The man on the phone: What's it like making history's highest auction bid?   Real      Fake
                   PM to states: start planning for vaccine rollout, maintain fatality rate below 1%   Real      Fake
                   DU survey shows Akhilesh-Maya more popular than Modi, Priyanka is a failed gambit   Real      Fake
                              UP registers first case under anti-conversion law in Bareilly district   Real      Fake
           Fact Check: Old Video Of Statues Of Ganesha Being Immersed Into River Falsely Linked With   Fake      Fake
         Special court rejects CBIÛªs closure report in IAS officerÛªs death, orders further probe   Real      Fake
                                  President Donald Trump has used this word over 250 times this year   Real      Fake
                                 From A Photoshopped Hea

In [46]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict using Logistic Regression
logistic_ifnd_pred = logistic_model.predict(ifnd_tfidf)

# Calculate accuracy
logistic_ifnd_accuracy = accuracy_score(
    ifnd_df["numeric_label"],
    logistic_ifnd_pred
)

print("=" * 60)
print("EXTERNAL VALIDATION - LOGISTIC REGRESSION ON IFND")
print("=" * 60)

print(f"\nACCURACY: {logistic_ifnd_accuracy:.6f}")
print(f"ACCURACY PERCENTAGE: {logistic_ifnd_accuracy * 100:.2f}%")

print("\nCLASSIFICATION REPORT:\n")
print(classification_report(
    ifnd_df["numeric_label"],
    logistic_ifnd_pred,
    target_names=["Fake", "Real"]
))

print("\nCONFUSION MATRIX:\n")
print(confusion_matrix(
    ifnd_df["numeric_label"],
    logistic_ifnd_pred
))

NameError: name 'ifnd_tfidf' is not defined

In [47]:
# Transform IFND statements using the existing TF-IDF vectorizer

ifnd_tfidf = tfidf_vectorizer.transform(
    ifnd_df["Statement"].astype(str)
)

print("IFND TF-IDF transformation completed!")
print("Shape:", ifnd_tfidf.shape)

NameError: name 'tfidf_vectorizer' is not defined

In [48]:
print([name for name in globals() if "tfidf" in name.lower() or "vector" in name.lower()])


['TfidfVectorizer', 'tfidf', 'X_train_tfidf', 'X_test_tfidf', 'X_ifnd_tfidf']


In [49]:
# Predict IFND dataset using Logistic Regression
logistic_ifnd_pred = logistic_model.predict(X_ifnd_tfidf)

# Calculate accuracy
logistic_ifnd_accuracy = accuracy_score(
    ifnd_df["numeric_label"],
    logistic_ifnd_pred
)

print("=" * 60)
print("EXTERNAL VALIDATION — LOGISTIC REGRESSION ON IFND")
print("=" * 60)

print(f"\nACCURACY: {logistic_ifnd_accuracy:.6f}")
print(f"ACCURACY PERCENTAGE: {logistic_ifnd_accuracy * 100:.2f}%\n")

print("CLASSIFICATION REPORT:\n")
print(classification_report(
    ifnd_df["numeric_label"],
    logistic_ifnd_pred,
    target_names=["Fake", "Real"]
))

print("CONFUSION MATRIX:\n")
print(confusion_matrix(
    ifnd_df["numeric_label"],
    logistic_ifnd_pred
))

EXTERNAL VALIDATION — LOGISTIC REGRESSION ON IFND

ACCURACY: 0.436153
ACCURACY PERCENTAGE: 43.62%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.37      0.97      0.53     18914
        Real       0.91      0.17      0.29     37800

    accuracy                           0.44     56714
   macro avg       0.64      0.57      0.41     56714
weighted avg       0.73      0.44      0.37     56714

CONFUSION MATRIX:

[[18304   610]
 [31368  6432]]


In [50]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict IFND dataset using Random Forest
rf_ifnd_pred = rf_model.predict(X_ifnd_tfidf)

# Calculate accuracy
rf_ifnd_accuracy = accuracy_score(
    ifnd_df["numeric_label"],
    rf_ifnd_pred
)

print("=" * 60)
print("EXTERNAL VALIDATION — RANDOM FOREST ON IFND")
print("=" * 60)

print(f"\nACCURACY: {rf_ifnd_accuracy:.6f}")
print(f"ACCURACY PERCENTAGE: {rf_ifnd_accuracy * 100:.2f}%\n")

print("CLASSIFICATION REPORT:\n")
print(classification_report(
    ifnd_df["numeric_label"],
    rf_ifnd_pred,
    target_names=["Fake", "Real"]
))

print("CONFUSION MATRIX:\n")
print(confusion_matrix(
    ifnd_df["numeric_label"],
    rf_ifnd_pred
))

EXTERNAL VALIDATION — RANDOM FOREST ON IFND

ACCURACY: 0.333198
ACCURACY PERCENTAGE: 33.32%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.33      0.99      0.50     18914
        Real       0.45      0.00      0.00     37800

    accuracy                           0.33     56714
   macro avg       0.39      0.50      0.25     56714
weighted avg       0.41      0.33      0.17     56714

CONFUSION MATRIX:

[[18814   100]
 [37717    83]]


In [51]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predict IFND dataset using Naive Bayes
nb_ifnd_pred = nb_model.predict(X_ifnd_tfidf)

# Calculate accuracy
nb_ifnd_accuracy = accuracy_score(
    ifnd_df["numeric_label"],
    nb_ifnd_pred
)

print("=" * 60)
print("EXTERNAL VALIDATION — NAIVE BAYES ON IFND")
print("=" * 60)

print(f"\nACCURACY: {nb_ifnd_accuracy:.6f}")
print(f"ACCURACY PERCENTAGE: {nb_ifnd_accuracy * 100:.2f}%\n")

print("CLASSIFICATION REPORT:\n")
print(classification_report(
    ifnd_df["numeric_label"],
    nb_ifnd_pred,
    target_names=["Fake", "Real"]
))

print("CONFUSION MATRIX:\n")
print(confusion_matrix(
    ifnd_df["numeric_label"],
    nb_ifnd_pred
))

EXTERNAL VALIDATION — NAIVE BAYES ON IFND

ACCURACY: 0.746130
ACCURACY PERCENTAGE: 74.61%

CLASSIFICATION REPORT:

              precision    recall  f1-score   support

        Fake       0.59      0.75      0.66     18914
        Real       0.86      0.74      0.80     37800

    accuracy                           0.75     56714
   macro avg       0.73      0.75      0.73     56714
weighted avg       0.77      0.75      0.75     56714

CONFUSION MATRIX:

[[14200  4714]
 [ 9684 28116]]


In [52]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import pandas as pd

# Different TF-IDF configurations to test
tfidf_configs = [
    
    {
        "name": "Baseline",
        "ngram_range": (1, 1),
        "min_df": 1,
        "max_df": 1.0,
        "sublinear_tf": False
    },
    
    {
        "name": "Unigram + Bigram",
        "ngram_range": (1, 2),
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True
    },
    
    {
        "name": "Unigram + Bigram Optimized",
        "ngram_range": (1, 2),
        "min_df": 3,
        "max_df": 0.90,
        "sublinear_tf": True
    },
    
    {
        "name": "Unigram + Bigram + Trigram",
        "ngram_range": (1, 3),
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True
    }
]

results = []

for config in tfidf_configs:
    
    print(f"\nTesting: {config['name']}")
    
    # Create vectorizer
    vectorizer = TfidfVectorizer(
        ngram_range=config["ngram_range"],
        min_df=config["min_df"],
        max_df=config["max_df"],
        sublinear_tf=config["sublinear_tf"],
        max_features=50000
    )
    
    # Fit ONLY on training data
    X_train_vec = vectorizer.fit_transform(X_train)
    
    # Transform test data
    X_test_vec = vectorizer.transform(X_test)
    
    # Temporary baseline model for comparing TF-IDF
    model = LinearSVC()
    
    model.fit(X_train_vec, y_train)
    
    predictions = model.predict(X_test_vec)
    
    accuracy = accuracy_score(y_test, predictions)
    
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    results.append({
        "Configuration": config["name"],
        "Accuracy": accuracy * 100,
        "Features": X_train_vec.shape[1]
    })


# Display comparison
results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("TF-IDF CONFIGURATION COMPARISON")
print("="*60)

print(
    results_df
    .sort_values(by="Accuracy", ascending=False)
    .to_string(index=False)
)


Testing: Baseline
Accuracy: 99.42%

Testing: Unigram + Bigram
Accuracy: 99.68%

Testing: Unigram + Bigram Optimized
Accuracy: 99.68%

Testing: Unigram + Bigram + Trigram
Accuracy: 99.68%

TF-IDF CONFIGURATION COMPARISON
             Configuration  Accuracy  Features
          Unigram + Bigram 99.680307     50000
Unigram + Bigram Optimized 99.680307     50000
Unigram + Bigram + Trigram 99.680307     50000
                  Baseline 99.424552     50000


In [53]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

# ============================================================
# CREATE THE SELECTED TF-IDF REPRESENTATION
# ============================================================

tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    max_features=50000
)

# Fit TF-IDF on training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)

# Transform test data using the same vocabulary
X_test_tfidf = tfidf_vectorizer.transform(X_test)


# ============================================================
# TEST DIFFERENT NAIVE BAYES ALPHA VALUES
# ============================================================

alpha_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

nb_results = []

for alpha in alpha_values:

    print(f"\nTesting Naive Bayes with alpha = {alpha}")

    # Create Naive Bayes model
    nb_model = MultinomialNB(alpha=alpha)

    # Train the model
    nb_model.fit(X_train_tfidf, y_train)

    # Make predictions
    nb_predictions = nb_model.predict(X_test_tfidf)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, nb_predictions)

    print(f"Accuracy: {accuracy * 100:.2f}%")

    # Save results
    nb_results.append({
        "Alpha": alpha,
        "Accuracy": accuracy * 100
    })


# ============================================================
# DISPLAY RESULTS
# ============================================================

nb_results_df = pd.DataFrame(nb_results)

nb_results_df = nb_results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n" + "=" * 55)
print("NAIVE BAYES HYPERPARAMETER TUNING RESULTS")
print("=" * 55)

print(nb_results_df.to_string(index=False))


# ============================================================
# FIND THE BEST ALPHA
# ============================================================

best_alpha = nb_results_df.iloc[0]["Alpha"]

print("\nBEST ALPHA:", best_alpha)


Testing Naive Bayes with alpha = 0.001
Accuracy: 96.55%

Testing Naive Bayes with alpha = 0.01
Accuracy: 96.36%

Testing Naive Bayes with alpha = 0.05
Accuracy: 96.28%

Testing Naive Bayes with alpha = 0.1
Accuracy: 96.21%

Testing Naive Bayes with alpha = 0.5
Accuracy: 95.97%

Testing Naive Bayes with alpha = 1.0
Accuracy: 95.88%

Testing Naive Bayes with alpha = 2.0
Accuracy: 95.83%

Testing Naive Bayes with alpha = 5.0
Accuracy: 95.45%

Testing Naive Bayes with alpha = 10.0
Accuracy: 95.26%

NAIVE BAYES HYPERPARAMETER TUNING RESULTS
 Alpha  Accuracy
 0.001 96.547315
 0.010 96.355499
 0.050 96.278772
 0.100 96.214834
 0.500 95.971867
 1.000 95.882353
 2.000 95.831202
 5.000 95.447570
10.000 95.255754

BEST ALPHA: 0.001


In [54]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score
import pandas as pd

# ============================================================
# TUNE LINEAR SVM
# ============================================================

# Different C values to test
c_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]

svm_results = []

for c in c_values:

    print(f"\nTesting Linear SVM with C = {c}")

    # Create Linear SVM model
    svm_model = LinearSVC(
        C=c,
        max_iter=10000
    )

    # Train the model
    svm_model.fit(X_train_tfidf, y_train)

    # Make predictions
    svm_predictions = svm_model.predict(X_test_tfidf)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, svm_predictions)

    print(f"Accuracy: {accuracy * 100:.2f}%")

    # Save result
    svm_results.append({
        "C": c,
        "Accuracy": accuracy * 100
    })


# ============================================================
# DISPLAY RESULTS
# ============================================================

svm_results_df = pd.DataFrame(svm_results)

svm_results_df = svm_results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n" + "=" * 55)
print("LINEAR SVM HYPERPARAMETER TUNING RESULTS")
print("=" * 55)

print(svm_results_df.to_string(index=False))


# ============================================================
# FIND THE BEST C VALUE
# ============================================================

best_c = svm_results_df.iloc[0]["C"]

print("\nBEST C:", best_c)


Testing Linear SVM with C = 0.001
Accuracy: 92.51%

Testing Linear SVM with C = 0.01
Accuracy: 97.71%

Testing Linear SVM with C = 0.05
Accuracy: 99.10%

Testing Linear SVM with C = 0.1
Accuracy: 99.41%

Testing Linear SVM with C = 0.5
Accuracy: 99.64%

Testing Linear SVM with C = 1
Accuracy: 99.68%

Testing Linear SVM with C = 2
Accuracy: 99.67%

Testing Linear SVM with C = 5
Accuracy: 99.71%

Testing Linear SVM with C = 10
Accuracy: 99.71%

LINEAR SVM HYPERPARAMETER TUNING RESULTS
     C  Accuracy
 5.000 99.705882
10.000 99.705882
 1.000 99.680307
 2.000 99.667519
 0.500 99.641944
 0.100 99.411765
 0.050 99.104859
 0.010 97.710997
 0.001 92.506394

BEST C: 5.0


In [55]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import pandas as pd

# ============================================================
# TUNE LOGISTIC REGRESSION
# ============================================================

# Different C values to test
c_values = [0.001, 0.01, 0.05, 0.1, 0.5, 1, 2, 5, 10]

lr_results = []

for c in c_values:

    print(f"\nTesting Logistic Regression with C = {c}")

    # Create the Logistic Regression model
    lr_model = LogisticRegression(
        C=c,
        max_iter=5000,
        solver="liblinear"
    )

    # Train the model
    lr_model.fit(X_train_tfidf, y_train)

    # Make predictions
    lr_predictions = lr_model.predict(X_test_tfidf)

    # Calculate accuracy
    accuracy = accuracy_score(y_test, lr_predictions)

    print(f"Accuracy: {accuracy * 100:.2f}%")

    # Save the result
    lr_results.append({
        "C": c,
        "Accuracy": accuracy * 100
    })


# ============================================================
# DISPLAY RESULTS
# ============================================================

lr_results_df = pd.DataFrame(lr_results)

lr_results_df = lr_results_df.sort_values(
    by="Accuracy",
    ascending=False
)

print("\n" + "=" * 55)
print("LOGISTIC REGRESSION HYPERPARAMETER TUNING RESULTS")
print("=" * 55)

print(lr_results_df.to_string(index=False))


# ============================================================
# FIND THE BEST C VALUE
# ============================================================

best_lr_c = lr_results_df.iloc[0]["C"]

print("\nBEST C:", best_lr_c)


Testing Logistic Regression with C = 0.001
Accuracy: 54.45%

Testing Logistic Regression with C = 0.01
Accuracy: 93.53%

Testing Logistic Regression with C = 0.05
Accuracy: 96.62%

Testing Logistic Regression with C = 0.1
Accuracy: 97.51%

Testing Logistic Regression with C = 0.5
Accuracy: 98.77%

Testing Logistic Regression with C = 1
Accuracy: 99.05%

Testing Logistic Regression with C = 2
Accuracy: 99.28%

Testing Logistic Regression with C = 5
Accuracy: 99.46%

Testing Logistic Regression with C = 10
Accuracy: 99.50%

LOGISTIC REGRESSION HYPERPARAMETER TUNING RESULTS
     C  Accuracy
10.000 99.501279
 5.000 99.462916
 2.000 99.283887
 1.000 99.053708
 0.500 98.772379
 0.100 97.506394
 0.050 96.624041
 0.010 93.529412
 0.001 54.450128

BEST C: 10.0


In [56]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import pandas as pd

# Different Random Forest configurations to test
rf_configs = [
    
    {
        "name": "RF Configuration 1",
        "n_estimators": 100,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1
    },
    
    {
        "name": "RF Configuration 2",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 2,
        "min_samples_leaf": 1
    },
    
    {
        "name": "RF Configuration 3",
        "n_estimators": 200,
        "max_depth": 50,
        "min_samples_split": 2,
        "min_samples_leaf": 1
    },
    
    {
        "name": "RF Configuration 4",
        "n_estimators": 300,
        "max_depth": 100,
        "min_samples_split": 2,
        "min_samples_leaf": 1
    },
    
    {
        "name": "RF Configuration 5",
        "n_estimators": 200,
        "max_depth": None,
        "min_samples_split": 5,
        "min_samples_leaf": 1
    }
]

results = []

for config in rf_configs:
    
    print(f"\nTesting: {config['name']}")
    
    # Create Random Forest with current configuration
    rf_model = RandomForestClassifier(
        n_estimators=config["n_estimators"],
        max_depth=config["max_depth"],
        min_samples_split=config["min_samples_split"],
        min_samples_leaf=config["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )
    
    # Train the model
    rf_model.fit(X_train_vec, y_train)
    
    # Make predictions
    predictions = rf_model.predict(X_test_vec)
    
    # Calculate accuracy
    accuracy = accuracy_score(y_test, predictions)
    
    print(f"Accuracy: {accuracy * 100:.2f}%")
    
    # Save results
    results.append({
        "Configuration": config["name"],
        "n_estimators": config["n_estimators"],
        "max_depth": config["max_depth"],
        "min_samples_split": config["min_samples_split"],
        "min_samples_leaf": config["min_samples_leaf"],
        "Accuracy (%)": accuracy * 100
    })


# Convert results into a DataFrame
rf_results_df = pd.DataFrame(results)

# Sort by best accuracy
rf_results_df = rf_results_df.sort_values(
    by="Accuracy (%)",
    ascending=False
)

print("\n" + "=" * 80)
print("RANDOM FOREST TUNING RESULTS")
print("=" * 80)

print(rf_results_df.to_string(index=False))

# Display the best configuration
best_rf = rf_results_df.iloc[0]

print("\n" + "=" * 80)
print("BEST RANDOM FOREST CONFIGURATION")
print("=" * 80)

print(best_rf)


Testing: RF Configuration 1
Accuracy: 99.36%

Testing: RF Configuration 2
Accuracy: 99.48%

Testing: RF Configuration 3
Accuracy: 99.27%

Testing: RF Configuration 4
Accuracy: 99.51%

Testing: RF Configuration 5
Accuracy: 99.39%

RANDOM FOREST TUNING RESULTS
     Configuration  n_estimators  max_depth  min_samples_split  min_samples_leaf  Accuracy (%)
RF Configuration 4           300      100.0                  2                 1     99.514066
RF Configuration 2           200        NaN                  2                 1     99.475703
RF Configuration 5           200        NaN                  5                 1     99.386189
RF Configuration 1           100        NaN                  2                 1     99.360614
RF Configuration 3           200       50.0                  2                 1     99.271100

BEST RANDOM FOREST CONFIGURATION
Configuration        RF Configuration 4
n_estimators                        300
max_depth                         100.0
min_samples_spli

In [57]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import pandas as pd


# ==========================================
# CREATE FINAL TUNED MODELS
# ==========================================

models = {

    "Naive Bayes": MultinomialNB(
        alpha=0.001
    ),

    "Linear SVM": LinearSVC(
        C=5
    ),

    "Logistic Regression": LogisticRegression(
        C=10,
        max_iter=2000
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        max_depth=100,
        min_samples_split=2,
        min_samples_leaf=1,
        random_state=42,
        n_jobs=-1
    )
}


# ==========================================
# STORE FINAL RESULTS
# ==========================================

final_results = []


# ==========================================
# TRAIN AND EVALUATE EVERY MODEL
# ==========================================

for name, model in models.items():

    print("\n" + "=" * 60)
    print(f"TRAINING: {name}")
    print("=" * 60)

    # Train model
    model.fit(X_train_vec, y_train)

    # Predict on SAME untouched test set
    predictions = model.predict(X_test_vec)

    # Calculate metrics
    accuracy = accuracy_score(y_test, predictions)

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    # Display results
    print(f"Accuracy:  {accuracy * 100:.4f}%")
    print(f"Precision: {precision * 100:.4f}%")
    print(f"Recall:    {recall * 100:.4f}%")
    print(f"F1 Score:  {f1 * 100:.4f}%")

    # Save results
    final_results.append({

        "Model": name,

        "Accuracy (%)": accuracy * 100,

        "Precision (%)": precision * 100,

        "Recall (%)": recall * 100,

        "F1 Score (%)": f1 * 100
    })


# ==========================================
# CREATE FINAL COMPARISON TABLE
# ==========================================

final_comparison = pd.DataFrame(final_results)


# Sort by F1 Score first, then Accuracy
final_comparison = final_comparison.sort_values(
    by=["F1 Score (%)", "Accuracy (%)"],
    ascending=False
)


print("\n\n" + "=" * 75)
print("🏆 FINAL INTERNAL MODEL COMPARISON")
print("=" * 75)

print(
    final_comparison.to_string(
        index=False
    )
)


TRAINING: Naive Bayes
Accuracy:  97.0716%
Precision: 97.0735%
Recall:    97.0716%
F1 Score:  97.0704%

TRAINING: Linear SVM
Accuracy:  99.7059%
Precision: 99.7060%
Recall:    99.7059%
F1 Score:  99.7059%

TRAINING: Logistic Regression
Accuracy:  99.4373%
Precision: 99.4375%
Recall:    99.4373%
F1 Score:  99.4373%

TRAINING: Random Forest
Accuracy:  99.5141%
Precision: 99.5142%
Recall:    99.5141%
F1 Score:  99.5140%


🏆 FINAL INTERNAL MODEL COMPARISON
              Model  Accuracy (%)  Precision (%)  Recall (%)  F1 Score (%)
         Linear SVM     99.705882      99.705998   99.705882     99.705860
      Random Forest     99.514066      99.514191   99.514066     99.514024
Logistic Regression     99.437340      99.437544   99.437340     99.437278
        Naive Bayes     97.071611      97.073511   97.071611     97.070370


In [58]:
print([name for name in globals() 
       if "tfidf" in name.lower() or "vector" in name.lower()])

['TfidfVectorizer', 'tfidf', 'X_train_tfidf', 'X_test_tfidf', 'X_ifnd_tfidf', 'tfidf_configs', 'vectorizer', 'tfidf_vectorizer']


In [59]:
print([
    name for name in globals()
    if any(keyword in name.lower() for keyword in [
        "svm", "logistic", "random", "forest", "naive", "bayes", "model"
    ])
])

['LogisticRegression', 'model', 'nb_model', 'svm_model', 'svm_pred', 'svm_accuracy', 'logistic_model', 'logistic_pred', 'logistic_accuracy', 'cv_svm', 'RandomForestClassifier', 'rf_model', 'logistic_ifnd_pred', 'logistic_ifnd_accuracy', 'svm_results', 'svm_predictions', 'svm_results_df', 'lr_model', 'models']


In [60]:
# ==========================================
# FINAL IFND DATA PREPARATION
# ==========================================

# Transform IFND statements using the TF-IDF
# vectorizer that was fitted ONLY on training data

X_ifnd_final = tfidf_vectorizer.transform(
    ifnd_df["Statement"].astype(str)
)

# True IFND labels
y_ifnd_final = ifnd_df["numeric_label"]

print("IFND data prepared successfully!")
print("TF-IDF shape:", X_ifnd_final.shape)
print("\nLabel distribution:")
print(y_ifnd_final.value_counts())

IFND data prepared successfully!
TF-IDF shape: (56714, 50000)

Label distribution:
numeric_label
1    37800
0    18914
Name: count, dtype: int64


In [61]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

import pandas as pd

# ==========================================
# FINAL UNTOUCHED IFND EVALUATION
# ==========================================

final_models = {
    "Naive Bayes": nb_model,
    "Linear SVM": svm_model,
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model
}

final_results = []

for model_name, trained_model in final_models.items():

    print("\n" + "=" * 65)
    print(f"FINAL IFND EVALUATION — {model_name}")
    print("=" * 65)

    # Make predictions on untouched IFND data
    predictions = trained_model.predict(X_ifnd_final)

    # Calculate metrics
    accuracy = accuracy_score(y_ifnd_final, predictions)

    precision = precision_score(
        y_ifnd_final,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_ifnd_final,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_ifnd_final,
        predictions,
        average="weighted",
        zero_division=0
    )

    # Store results
    final_results.append({
        "Model": model_name,
        "Accuracy (%)": accuracy * 100,
        "Precision (%)": precision * 100,
        "Recall (%)": recall * 100,
        "F1 Score (%)": f1 * 100
    })

    # Display metrics
    print(f"\nAccuracy:  {accuracy * 100:.2f}%")
    print(f"Precision: {precision * 100:.2f}%")
    print(f"Recall:    {recall * 100:.2f}%")
    print(f"F1 Score:  {f1 * 100:.2f}%")

    # Classification report
    print("\nCLASSIFICATION REPORT:")
    print(
        classification_report(
            y_ifnd_final,
            predictions,
            target_names=["Fake", "Real"],
            zero_division=0
        )
    )

    # Confusion matrix
    print("CONFUSION MATRIX:")
    print(confusion_matrix(y_ifnd_final, predictions))


# ==========================================
# FINAL MODEL COMPARISON
# ==========================================

final_results_df = pd.DataFrame(final_results)

final_results_df = final_results_df.sort_values(
    by="Accuracy (%)",
    ascending=False
).reset_index(drop=True)

print("\n" + "=" * 65)
print("🏆 FINAL UNTOUCHED IFND MODEL COMPARISON")
print("=" * 65)

print(final_results_df.to_string(index=False))


FINAL IFND EVALUATION — Naive Bayes

Accuracy:  79.42%
Precision: 79.71%
Recall:    79.42%
F1 Score:  79.54%

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

        Fake       0.68      0.72      0.70     18914
        Real       0.85      0.83      0.84     37800

    accuracy                           0.79     56714
   macro avg       0.77      0.77      0.77     56714
weighted avg       0.80      0.79      0.80     56714

CONFUSION MATRIX:
[[13560  5354]
 [ 6319 31481]]

FINAL IFND EVALUATION — Linear SVM

Accuracy:  38.76%
Precision: 71.58%
Recall:    38.76%
F1 Score:  28.28%

CLASSIFICATION REPORT:
              precision    recall  f1-score   support

        Fake       0.35      0.98      0.52     18914
        Real       0.90      0.09      0.17     37800

    accuracy                           0.39     56714
   macro avg       0.62      0.54      0.34     56714
weighted avg       0.72      0.39      0.28     56714

CONFUSION MATRIX:
[[18524   39

In [62]:
print("=" * 70)
print("MODEL FEATURE REQUIREMENTS")
print("=" * 70)

models_to_check = {
    "Naive Bayes": nb_model,
    "Linear SVM": svm_model,
    "Logistic Regression": logistic_model,
    "Random Forest": rf_model
}

for name, trained_model in models_to_check.items():

    print(f"\n{name}")

    if hasattr(trained_model, "n_features_in_"):
        print("Expected features:", trained_model.n_features_in_)
    else:
        print("Expected features: Not directly available")

print("\n" + "=" * 70)
print("CURRENT DATA MATRICES")
print("=" * 70)

print("X_train_tfidf shape:", X_train_tfidf.shape)
print("X_test_tfidf shape: ", X_test_tfidf.shape)
print("X_ifnd_tfidf shape: ", X_ifnd_tfidf.shape)

print("\nX_ifnd_final shape:", X_ifnd_final.shape)

MODEL FEATURE REQUIREMENTS

Naive Bayes
Expected features: 50000

Linear SVM
Expected features: 50000

Logistic Regression
Expected features: 50000

Random Forest
Expected features: 50000

CURRENT DATA MATRICES
X_train_tfidf shape: (31276, 50000)
X_test_tfidf shape:  (7820, 50000)
X_ifnd_tfidf shape:  (56714, 50000)

X_ifnd_final shape: (56714, 50000)


In [63]:
from sklearn.feature_extraction.text import TfidfVectorizer

# ==========================================
# FINAL TF-IDF REPRESENTATION
# ==========================================

final_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    sublinear_tf=True,
    max_features=50000
)

# Fit ONLY on training data
X_train_final = final_vectorizer.fit_transform(
    X_train.astype(str)
)

# Transform internal test data
X_test_final = final_vectorizer.transform(
    X_test.astype(str)
)

# Transform untouched IFND data
# IMPORTANT: transform(), NOT fit_transform()
X_ifnd_final = final_vectorizer.transform(
    ifnd_df["Statement"].astype(str)
)

print("FINAL TF-IDF REPRESENTATION CREATED SUCCESSFULLY!")

print("\nTraining shape:", X_train_final.shape)
print("Test shape:", X_test_final.shape)
print("IFND shape:", X_ifnd_final.shape)

FINAL TF-IDF REPRESENTATION CREATED SUCCESSFULLY!

Training shape: (31276, 50000)
Test shape: (7820, 50000)
IFND shape: (56714, 50000)
